# Sasa & Dany · Milestone 1 · Pre-cleaning profile

**What this notebook is:** the exploratory data analysis (EDA) we do *before* any cleaning, for 11 of the 14 tables:
`seasons`, `circuits`, `drivers`, `constructors`, `status`, `sprint_results`, `driver_standings`,
`lap_times`, `pit_stops`, `constructor_standings`, `constructor_results`.

**Why:** the brief asks for EDA before cleaning, backed by code and plots instead of looking at rows by eye.
What we find here decides how each table is cleaned later, and feeds the data-engineering questions,
the features and the report.

**One rule for the whole notebook:** the raw files are never changed. Every step works on copies.

## P0 · Setup: load untouched, fingerprint, build helpers

- **What:** load our 11 CSV files exactly as delivered (every column as text), save a fingerprint of each one, and define four helper functions.
- **Why:**
  - Reading everything as text keeps the `\N` placeholders visible, so we can count them instead of letting pandas hide them.
  - The fingerprint lets us prove at the end (step D3) that the raw data was never modified, as the brief requires.
  - The helpers (`as_num`, `year_of`, `profile`, `pk_check`) make every later step use the same checks, so our EDA is code, not manual inspection.

In [1]:
# ==========================================================================================
# P0 · SETUP: load our 11 tables exactly as they are, and build the helpers every step uses
# ==========================================================================================

# ---------- 1. Libraries ----------
import os                          # to find the folder where Kaggle put the dataset
import numpy as np                 # gives us NaN, the real "missing value" marker
import pandas as pd                # tables (DataFrames): reading, counting, grouping
import matplotlib.pyplot as plt    # basic plots
import seaborn as sns              # nicer statistical plots (heatmaps, etc.)

pd.set_option("display.max_columns", 50)   # show every column when we display a table


# ---------- 2. Where is the data? ----------
# Kaggle puts attached datasets somewhere under /kaggle/input. Instead of guessing the exact
# folder name, we search for the folder that contains races.csv. On a laptop we fall back to ./data
def find_data_dir():
    for folder, _, files in os.walk("/kaggle/input"):
        if "races.csv" in files:
            return folder
    return "data"

DATA_DIR = find_data_dir()
csv_files = sorted(f for f in os.listdir(DATA_DIR) if f.endswith(".csv"))
print("Data folder:", DATA_DIR)
print("CSV files found:", len(csv_files))          # we expect 14


# ---------- 3. The 11 tables we (Sasa & Dany) profile ----------
# races, results and qualifying are profiled by our teammates; we only read them for lookups.
MINE = ["seasons", "circuits", "drivers", "constructors", "status", "sprint_results",
        "driver_standings", "lap_times", "pit_stops", "constructor_standings", "constructor_results"]

# The dataset does not leave missing cells empty: it writes the text \N instead.
# A value that "stands in" for missing like this is called a sentinel.
SENT = r"\N"


# ---------- 4. Load every file exactly as it is (the "bronze" layer) ----------
# dtype=str              -> read every column as text, so pandas does not guess the types for us
# keep_default_na=False  -> pandas must not turn anything into NaN by itself, so \N stays visible
raw = {t: pd.read_csv(f"{DATA_DIR}/{t}.csv", dtype=str, keep_default_na=False)
       for t in MINE + ["races", "results"]}

# A fingerprint (hash) of every raw table. In step D3 we compute it again and compare,
# to PROVE we never changed the raw data (the brief: "Raw CSV files remain immutable").
RAW_HASH = {t: pd.util.hash_pandas_object(df).sum() for t, df in raw.items()}


# ---------- 5. Two lookups we need in many steps ----------
# races: turn \N into NaN and make ids and year numeric, so we can find the season of any raceId
races = raw["races"].replace(SENT, np.nan).astype({"raceId": int, "year": int, "round": int, "circuitId": int})
races["date"] = pd.to_datetime(races["date"])      # race date as a real date, not text
YEAR = races.set_index("raceId")["year"]           # quick lookup: raceId -> season year
results = raw["results"].replace(SENT, np.nan)     # results with \N as NaN, used only for cross-checks


# ---------- 6. Helper functions (the same checks for every table) ----------
def as_num(df):
    # Returns a COPY for exploring: \N becomes NaN, and every column made only of numbers becomes numeric.
    # Columns that contain any text (e.g. lap times like "1:26.572") stay text, so we can still see their format.
    out = df.replace(SENT, np.nan)
    for c in out.columns:
        conv = pd.to_numeric(out[c], errors="coerce")     # try to read the column as numbers
        if conv.notna().sum() == out[c].notna().sum():    # did EVERY non-missing value convert?
            out[c] = conv                                 # yes -> keep the numeric version
    return out


def year_of(df):
    # The season year of every row, for any table that has a raceId column
    return pd.to_numeric(df["raceId"]).map(YEAR)


def profile(t):
    # A "profile card" for table t: one row per column, with the checks we care about
    df = raw[t]
    return pd.DataFrame({
        "sentinel_N": df.eq(SENT).sum(),                     # how many \N in the column
        "sentinel_%": (df.eq(SENT).mean() * 100).round(2),   # the same, as a % of the rows
        "empty_str": df.eq("").sum(),                        # truly empty cells (we expect 0)
        "unique": df.nunique(),                              # how many different values
        "numeric_%": df.apply(lambda s: pd.to_numeric(s, errors="coerce").notna().mean() * 100).round(1),  # % that read as numbers
        "example": df.iloc[0]})                              # the first value, to see the format


def pk_check(t, cols):
    # Checks a primary key: the column(s) that should identify each row of table t
    df = raw[t]
    return {"table": t, "key": "+".join(cols), "rows": len(df),
            "duplicates": int(df.duplicated(cols).sum()),                          # rows whose key appears again
            "null_or_sentinel": int(df[cols].isin(["", SENT]).any(axis=1).sum())}  # rows with a missing key


# ---------- 7. First output: how big is each of our 11 tables? ----------
sizes = pd.DataFrame({t: raw[t].shape for t in MINE}, index=["rows", "columns"]).T
sizes

Data folder: /kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020
CSV files found: 14


,rows,columns
seasons,75,2
circuits,77,9
drivers,861,9
constructors,212,5
status,139,2
sprint_results,360,16
driver_standings,34863,7
lap_times,589081,6
pit_stops,11371,7
constructor_standings,13391,7


**P0 result:** the dataset folder holds all 14 CSV files, and our 11 tables loaded with the sizes above.
`lap_times` is by far the biggest (589,081 rows); `seasons` and `status` are tiny lookup lists.
Nothing has been cleaned or changed; this is the starting point every later step builds on.